In [1]:
import destruction_models as models
from tensorflow.keras import callbacks, metrics
from tensorflow.keras.utils import Sequence
from destruction_utilities import *
import matplotlib.pyplot as plt
import tensorflow as tf
import time
import pickle

In [2]:
CITY = 'aleppo'
BATCH_SIZE = 32
TILE_SIZE = [128,128]

In [3]:
class SiameseGenerator(Sequence):
    def __init__(self, images, labels, batch_size=BATCH_SIZE):
        self.images_t0 = images[0]
        self.images_tt = images[1]
        self.labels = labels
        self.batch_size = batch_size
        self.tuple_pairs = make_tuple_pair(self.images_t0.shape[0], self.batch_size)
        np.random.shuffle(self.tuple_pairs)
   
    def __len__(self):
        return len(self.images_t0)//self.batch_size
    
    def __getitem__(self, index):

        index_range = self.tuple_pairs[index]
        indices = np.arange(0,32)
        np.random.shuffle(indices)
        
        X_t0 = self.images_t0[index_range[0]:index_range[1]][indices]
        X_t1 = self.images_tt[index_range[0]:index_range[1]][indices]
        y = self.labels[index_range[0]:index_range[1]][indices]
        
        return {'images_t0':X_t0, 'images_tt':X_t1}, y

# class SiameseGenerator(Sequence):
#     def __init__(self, images, labels, batch_size=BATCH_SIZE):
#         self.images_t0 = images[0]
#         self.images_tt = images[1]
#         self.labels = labels
#         self.batch_size = batch_size
        
#     def __len__(self):
#         return len(self.images_t0)//self.batch_size
    
#     def __getitem__(self, index):

#         X_t0 = self.images_t0[index*self.batch_size:(index+1)*self.batch_size]
#         X_t1 = self.images_tt[index*self.batch_size:(index+1)*self.batch_size]
#         y = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
#         return {'images_t0':X_t0, 'images_tt':X_t1}, y

In [4]:
train_images_t0 = read_zarr(CITY, 'images_siamese_train_t0')
train_images_tt = read_zarr(CITY, 'images_siamese_train_tt')
train_labels = read_zarr(CITY, 'labels_siamese_train')

valid_images_t0 = read_zarr(CITY, 'images_siamese_valid_t0')
valid_images_tt = read_zarr(CITY, 'images_siamese_valid_tt')
valid_labels = read_zarr(CITY, 'labels_siamese_valid')

train_gen = SiameseGenerator((train_images_t0, train_images_tt), train_labels)
valid_gen = SiameseGenerator((valid_images_t0, valid_images_tt), valid_labels)


In [5]:
def run_model(train_images, train_labels, valid_images, valid_labels):
    train_gen = SiameseGenerator((train_images[0], train_images[1]), train_labels)
    valid_gen = SiameseGenerator((valid_images[0], valid_images[1]), valid_labels)
    
    training_callbacks = [
        callbacks.EarlyStopping(monitor='val_auc', patience=15, restore_best_weights=True)
    ]
    
    filters = random.choice([16, 24, 32])
    dropout = random.choice(np.linspace(0.1, 0.2))
    epochs = random.choice(np.arange(70,100))
    units = random.choice([24, 32, 48])
    lr = random.choice([0.003, 0.01, 0.03, 0.1, 0.3])

    args_encode  = dict(filters=filters, dropout=dropout) # ! Check parameters before run
    args_dense  = dict(units=units, dropout=dropout) # ! Check parameters before run

    parameters = f'filters={filters}, \ndropout={np.round(dropout, 4)}, \nepochs={epochs}, \nunits={units}, \nlearning_rate={lr}'
    print(parameters)
    
#     model = models.siamese_convolutional_network(shape=(*TILE_SIZE, 3), args_encode=args_encode, args_dense=args_dense)
    model = models.siamese_convolutional_network(shape=(*TILE_SIZE, 3),  args_encode=dict(filters=8, dropout=0), args_dense=dict(units=16, dropout=0))
   
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy',metrics.AUC(num_thresholds=200, curve='ROC', name='auc')])
    model.summary()
    

    # Train model on dataset
    history = model.fit_generator(
        train_gen,
        validation_data=valid_gen,
        epochs=epochs,
        verbose=1,
        callbacks=training_callbacks
    )
    
    return model, history, parameters

In [6]:
def plot_training(H, P, ts, plotPath):
	# construct a plot that plots and saves the training history
	plt.style.use("ggplot")
	plt.figure()
	plt.plot(H.history["accuracy"], label="train_accuracy")
	plt.plot(H.history["val_accuracy"], label="val_accuracy")
	plt.plot(H.history["auc"], label="train_auc")
	plt.plot(H.history["val_auc"], label="val_auc")
	plt.title(f"Training Accuracy and AUC")
	plt.suptitle(f"City = {CITY}") 
	plt.xlabel("Epoch #")
	plt.ylabel("AUC")
	plt.text(0.65, 0.18, P + f"\nmax(val_auc)={np.round(np.max(H.history['val_auc']), 4)}\ncode={ts}", fontsize=8, transform=plt.gcf().transFigure)
	plt.legend(loc="lower left")
	plt.savefig(plotPath)

In [ ]:
for i in range(0,5):
    m = run_model((train_images_t0, train_images_tt), train_labels, (valid_images_t0, valid_images_tt), valid_labels)
    model = m[0]
    history = m[1]
    parameters = m[2]
    print("Model optimization complete..\n\n")
    ts = str(np.round(time.time()))
    with open(f'../models/{CITY}_SNN_RUN{i}_{ts}_hist', 'wb') as file_pi:
        pickle.dump(history.history, file_pi)
    
    model.save(f'../models/{CITY}_SNN_RUN{i}_{ts}', save_format="h5")
    plot_training(history, parameters, ts, f'../figures/{CITY}_SNN_RUN{i}_{ts}.png')
    
    with open('../models/run_parameters.txt', "a") as file:
        file.write(f'{CITY}_SNN_RUN{i}_{ts}: \n \t{parameters}\n')
      


filters=24, 
dropout=0.1, 
epochs=73, 
units=32, 
learning_rate=0.03
Metal device set to: Apple M1
Model: "siamese_convolutional_network"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 images_t0 (InputLayer)         [(None, 128, 128, 3  0           []                               
                                )]                                                                
                                                                                                  
 images_tt (InputLayer)         [(None, 128, 128, 3  0           []                               
                                )]                                                                
                                                                                                  
 encoder (Functional)           (None, 640)          23736       ['ima

2022-07-12 08:19:20.707102: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2022-07-12 08:19:20.707391: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/var/folders/z1/7pydgng971nfzdf2rwg_p__r0000gn/T/ipykernel_2444/1874604745.py:30: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history = model.fit_generator(


Epoch 1/73


2022-07-12 08:19:21.072363: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2022-07-12 08:19:21.800929: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


3210/3210 [==============================] - ETA: 0s - loss: 0.1550 - accuracy: 0.9625 - auc: 0.6607

...

3210/3210 [==============================] - 649s 202ms/step - loss: 0.1556 - accuracy: 0.9597 - auc: 0.8078 - val_loss: 187363804315648.0000 - val_accuracy: 0.9512 - val_auc: 0.6942
Epoch 44/83
3210/3210 [==============================] - 701s 218ms/step - loss: 0.1394 - accuracy: 0.9619 - auc: 0.8346 - val_loss: 1927820607488.0000 - val_accuracy: 0.8162 - val_auc: 0.7536
Epoch 45/83
3210/3210 [==============================] - 709s 221ms/step - loss: 0.1549 - accuracy: 0.9605 - auc: 0.8188 - val_loss: 17018517454848.0000 - val_accuracy: 0.9492 - val_auc: 0.7036
Model optimization complete..


filters=16, 
dropout=0.1122, 
epochs=89, 
units=48, 
learning_rate=0.1
Model: "siamese_convolutional_network"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 images_t0 (InputLayer)         [(None, 128, 128, 3  0           []                       

/var/folders/z1/7pydgng971nfzdf2rwg_p__r0000gn/T/ipykernel_2444/1874604745.py:30: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history = model.fit_generator(
2022-07-13 00:21:38.481039: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


3210/3210 [==============================] - ETA: 0s - loss: 0.1595 - accuracy: 0.9628 - auc: 0.6139

2022-07-13 00:32:48.301540: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


3210/3210 [==============================] - 726s 226ms/step - loss: 0.1595 - accuracy: 0.9628 - auc: 0.6139 - val_loss: 22.2328 - val_accuracy: 0.9554 - val_auc: 0.7561
Epoch 2/89
3210/3210 [==============================] - 724s 225ms/step - loss: 0.1589 - accuracy: 0.9628 - auc: 0.6258 - val_loss: 0.1461 - val_accuracy: 0.9689 - val_auc: 0.6687
Epoch 3/89
3210/3210 [==============================] - 743s 232ms/step - loss: 0.1582 - accuracy: 0.9630 - auc: 0.6407 - val_loss: 106.8332 - val_accuracy: 0.9645 - val_auc: 0.7074
Epoch 4/89
3210/3210 [==============================] - 756s 236ms/step - loss: 0.1564 - accuracy: 0.9628 - auc: 0.6543 - val_loss: 267322.0000 - val_accuracy: 0.9668 - val_auc: 0.7612
Epoch 5/89
3210/3210 [==============================] - 748s 233ms/step - loss: 0.1572 - accuracy: 0.9626 - auc: 0.6533 - val_loss: 55430.6641 - val_accuracy: 0.9680 - val_auc: 0.4995
Epoch 6/89
3210/3210 [==============================] - 743s 232ms/step - loss: 0.1571 - accuracy: 

```
filters=32, dropout=0.13163265306122449, epochs=77, units=32, learning_rate=0.01
filters=32, dropout=0.18571428571428572, epochs=88, units=48, learning_rate=0.003

Upto 88
filters=32, 
dropout=0.1939, 
epochs=77, 
units=32, 
learning_rate=0.012


```